# NB10a: Additional Features - Staffing and Quality Metrics

## Overview

This notebook extracts additional staffing and quality features from files already downloaded in the `data/raw/` directory, then merges them into the existing master dataset.

**Inputs:**
- POS File (Staffing data): Provider Number, bed counts, nursing staff, physician, allied health
- Hospital General Info (Quality metrics): Star ratings, measure comparisons (mortality, safety, readmission)
- Master dataset from NB06: Hospital with benchmarks

**Outputs:**
- Enriched hospital dataset with staffing ratios and quality features
- Summary statistics and coverage report

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'nb10a_additional_features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load POS file - staffing columns
pos_cols = ['PRVDR_NUM', 'BED_CNT', 'RN_CNT', 'RN_FLTM_CNT', 'RN_PRTM_CNT',
            'LPN_LVN_CNT', 'LPN_LVN_FLTM_CNT', 'LPN_LVN_PRTM_CNT',
            'PHYSN_CNT', 'PHYSN_ASTNT_CNT', 'RSDNT_PHYSN_CNT',
            'NRS_AIDE_FLTM_CNT', 'NRS_AIDE_PRTM_CNT',
            'PHRMCST_FLTM_CNT', 'PHRMCST_PRTM_CNT', 'REG_PHRMCST_CNT',
            'MDCL_SCL_WORKR_CNT', 'DIETN_CNT', 'SCL_WORKR_CNT',
            'SPCH_PTHLGST_CNT', 'OCPTNL_THRPST_CNT', 'PHYS_THRPST_CNT',
            'INHLTN_THRPST_CNT', 'MDCL_TCHNLGST_CNT',
            'DRCT_CARE_PRSNEL_CNT', 'PRSNEL_OTHR_CNT']

# Find the POS file
pos_path = None
for candidate in sorted(RAW_DIR.glob('POS_File_QIES*.csv'), reverse=True):
    if candidate.stat().st_size > 1_000_000:
        pos_path = candidate
        break

if pos_path is None:
    pos_path = RAW_DIR / 'pos_hospital.csv'

print(f'POS file: {pos_path.name}')

# Read only needed columns (handle missing columns gracefully)
df_pos_raw = pd.read_csv(pos_path, dtype=str, low_memory=False)
available_cols = [c for c in pos_cols if c in df_pos_raw.columns]
missing_cols = [c for c in pos_cols if c not in df_pos_raw.columns]
print(f'Available staffing columns: {len(available_cols)}/{len(pos_cols)}')
if missing_cols:
    print(f'Missing columns: {missing_cols}')

df_pos = df_pos_raw[available_cols].copy()

# Filter to hospitals (6-digit CCN starting with digits, not specialty codes)
df_pos['ccn'] = df_pos['PRVDR_NUM'].str.strip().str.zfill(6)

# Convert numeric columns
for col in available_cols:
    if col != 'PRVDR_NUM':
        df_pos[col] = pd.to_numeric(df_pos[col], errors='coerce')

print(f'POS records: {len(df_pos):,}')
print(f'\nSample staffing data:')
print(df_pos[['ccn', 'BED_CNT', 'RN_CNT', 'LPN_LVN_CNT', 'PHYSN_CNT']].head(10))

POS file: POS_File_QIES_Q4_2025.csv


Available staffing columns: 26/26


POS records: 77,522

Sample staffing data:
      ccn  BED_CNT  RN_CNT  LPN_LVN_CNT  PHYSN_CNT
0  010001    420.0  521.00        96.00      79.00
1  010004     64.0    9.00         4.00       1.00
2  010005    240.0  137.00        19.50       1.00
3  010006    338.0  415.00        13.00      30.00
4  010007     99.0   38.00        15.00       2.00
5  010008     65.0   33.87        16.07       0.85
6  010009    150.0   51.00        13.00       0.00
7  010010     90.0   90.00        13.00       0.00
8  010011    362.0  267.06        40.50       0.00
9  010012    134.0  121.05         3.90       4.00


## Feature Engineering from POS Staffing Data

Create staffing ratio features by normalizing staff counts per hospital bed. These ratios help control for hospital size differences when comparing quality outcomes.

In [2]:
# Create staffing ratio features (per bed)
beds = df_pos['BED_CNT'].replace(0, np.nan)

# Core staffing ratios
df_pos['rn_per_bed'] = df_pos['RN_CNT'] / beds
df_pos['lpn_per_bed'] = df_pos['LPN_LVN_CNT'] / beds
df_pos['physician_per_bed'] = df_pos['PHYSN_CNT'] / beds
df_pos['total_nursing_per_bed'] = (df_pos['RN_CNT'].fillna(0) + df_pos['LPN_LVN_CNT'].fillna(0)) / beds

# Physician assistant count if available
if 'PHYSN_ASTNT_CNT' in df_pos.columns:
    df_pos['pa_per_bed'] = df_pos['PHYSN_ASTNT_CNT'] / beds

# Total clinical staff (sum all available clinical roles)
clinical_cols = ['RN_CNT', 'LPN_LVN_CNT', 'PHYSN_CNT', 'PHYSN_ASTNT_CNT',
                 'PHRMCST_FLTM_CNT', 'PHRMCST_PRTM_CNT',
                 'SPCH_PTHLGST_CNT', 'OCPTNL_THRPST_CNT', 'PHYS_THRPST_CNT',
                 'INHLTN_THRPST_CNT', 'MDCL_TCHNLGST_CNT']
available_clinical = [c for c in clinical_cols if c in df_pos.columns]
df_pos['total_clinical_staff'] = df_pos[available_clinical].fillna(0).sum(axis=1)
df_pos['clinical_staff_per_bed'] = df_pos['total_clinical_staff'] / beds

# RN skill mix (proportion of nursing staff that are RNs vs LPNs)
total_nursing = df_pos['RN_CNT'].fillna(0) + df_pos['LPN_LVN_CNT'].fillna(0)
df_pos['rn_skill_mix'] = df_pos['RN_CNT'] / total_nursing.replace(0, np.nan)

# Staffing summary
staffing_features = ['rn_per_bed', 'lpn_per_bed', 'physician_per_bed', 
                     'total_nursing_per_bed', 'clinical_staff_per_bed', 'rn_skill_mix']
print('Staffing Feature Summary:')
for feat in staffing_features:
    if feat in df_pos.columns:
        valid = df_pos[feat].dropna()
        print(f'  {feat}: mean={valid.mean():.3f}, median={valid.median():.3f}, n={len(valid):,}')

Staffing Feature Summary:
  rn_per_bed: mean=0.486, median=0.150, n=24,713
  lpn_per_bed: mean=0.126, median=0.042, n=24,713
  physician_per_bed: mean=0.093, median=0.000, n=13,208
  total_nursing_per_bed: mean=0.595, median=0.250, n=25,427
  clinical_staff_per_bed: mean=0.745, median=0.267, n=25,427
  rn_skill_mix: mean=0.804, median=1.000, n=31,717


## Load Hospital General Info Quality Metrics

Extract quality metrics from CMS Hospital General Info file, including star ratings and performance comparisons across mortality, safety, and readmission measures.

In [3]:
# Load Hospital General Info for quality metrics
hgi_path = RAW_DIR / 'hospital_general_info.csv'
df_hgi = pd.read_csv(hgi_path, dtype=str)

print(f'Hospital General Info: {len(df_hgi):,} rows')
print(f'Columns: {df_hgi.columns.tolist()}')

# Create CCN
df_hgi['ccn'] = df_hgi['Facility ID'].str.strip().str.zfill(6)

# Parse star rating
df_hgi['star_rating'] = pd.to_numeric(df_hgi['Hospital overall rating'], errors='coerce')

# Parse quality measure counts
quality_cols_map = {
    'mort_measures_better': 'Count of MORT Measures Better',
    'mort_measures_worse': 'Count of MORT Measures Worse',
    'mort_measures_no_diff': 'Count of MORT Measures No Different',
    'mort_total_measures': 'Count of Facility MORT Measures',
    'safety_measures_better': 'Count of Safety Measures Better',
    'safety_measures_worse': 'Count of Safety Measures Worse',
    'safety_measures_no_diff': 'Count of Safety Measures No Different',
    'safety_total_measures': 'Count of Facility Safety Measures',
    'readm_measures_better': 'Count of READM Measures Better',
    'readm_measures_worse': 'Count of READM Measures Worse',
    'readm_measures_no_diff': 'Count of READM Measures No Different',
    'readm_total_measures': 'Count of Facility READM Measures',
}

for new_col, orig_col in quality_cols_map.items():
    if orig_col in df_hgi.columns:
        df_hgi[new_col] = pd.to_numeric(df_hgi[orig_col], errors='coerce')

# Create composite quality features
# Mortality performance: proportion of measures "better" out of total
df_hgi['mort_pct_better'] = (df_hgi['mort_measures_better'] / 
                              df_hgi['mort_total_measures'].replace(0, np.nan) * 100)
df_hgi['mort_pct_worse'] = (df_hgi['mort_measures_worse'] / 
                             df_hgi['mort_total_measures'].replace(0, np.nan) * 100)

# Safety performance
df_hgi['safety_pct_better'] = (df_hgi['safety_measures_better'] / 
                                df_hgi['safety_total_measures'].replace(0, np.nan) * 100)
df_hgi['safety_pct_worse'] = (df_hgi['safety_measures_worse'] / 
                               df_hgi['safety_total_measures'].replace(0, np.nan) * 100)

# Readmission performance  
df_hgi['readm_pct_better'] = (df_hgi['readm_measures_better'] / 
                               df_hgi['readm_total_measures'].replace(0, np.nan) * 100)
df_hgi['readm_pct_worse'] = (df_hgi['readm_measures_worse'] / 
                              df_hgi['readm_total_measures'].replace(0, np.nan) * 100)

# Overall quality score: (better - worse) across all domains
df_hgi['quality_score'] = (
    df_hgi['mort_measures_better'].fillna(0) + 
    df_hgi['safety_measures_better'].fillna(0) + 
    df_hgi['readm_measures_better'].fillna(0) -
    df_hgi['mort_measures_worse'].fillna(0) - 
    df_hgi['safety_measures_worse'].fillna(0) - 
    df_hgi['readm_measures_worse'].fillna(0)
)

print(f'\nStar Rating Distribution:')
print(df_hgi['star_rating'].value_counts().sort_index())
print(f'\nQuality Score: mean={df_hgi["quality_score"].mean():.2f}, median={df_hgi["quality_score"].median():.2f}')

Hospital General Info: 5,426 rows
Columns: ['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type', 'Hospital Ownership', 'Emergency Services', 'Meets criteria for birthing friendly designation', 'Hospital overall rating', 'Hospital overall rating footnote', 'MORT Group Measure Count', 'Count of Facility MORT Measures', 'Count of MORT Measures Better', 'Count of MORT Measures No Different', 'Count of MORT Measures Worse', 'MORT Group Footnote', 'Safety Group Measure Count', 'Count of Facility Safety Measures', 'Count of Safety Measures Better', 'Count of Safety Measures No Different', 'Count of Safety Measures Worse', 'Safety Group Footnote', 'READM Group Measure Count', 'Count of Facility READM Measures', 'Count of READM Measures Better', 'Count of READM Measures No Different', 'Count of READM Measures Worse', 'READM Group Footnote', 'Pt Exp Group Measure Count', 'Count of Facility Pt Exp Measures', 'Pt Exp Gro

## Merge and Save

Combine staffing and quality features with the master hospital dataset from NB06. Report coverage of new features across the full dataset.

In [4]:
# Prepare POS staffing features for merge
pos_output_cols = ['ccn', 'rn_per_bed', 'lpn_per_bed', 'physician_per_bed',
                   'total_nursing_per_bed', 'clinical_staff_per_bed', 'rn_skill_mix',
                   'total_clinical_staff', 'RN_CNT', 'LPN_LVN_CNT', 'PHYSN_CNT']
pos_output_cols = [c for c in pos_output_cols if c in df_pos.columns]
df_pos_out = df_pos[pos_output_cols].copy()

# Prepare quality features for merge
quality_output_cols = ['ccn', 'star_rating', 'mort_pct_better', 'mort_pct_worse',
                       'safety_pct_better', 'safety_pct_worse',
                       'readm_pct_better', 'readm_pct_worse', 'quality_score']
quality_output_cols = [c for c in quality_output_cols if c in df_hgi.columns]
df_quality_out = df_hgi[quality_output_cols].copy()

# Load existing master dataset from NB06
master_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb06_peer_benchmarks' / 'hospital_with_benchmarks.csv'
df_master = pd.read_csv(master_path, dtype={'ccn': str})
df_master['ccn'] = df_master['ccn'].str.zfill(6)

print(f'Master dataset: {len(df_master):,} hospitals')

# Merge staffing
df_merged = df_master.merge(df_pos_out, on='ccn', how='left')

# Merge quality (avoid duplicate columns)
quality_new_cols = [c for c in quality_output_cols if c not in df_merged.columns or c == 'ccn']
df_merged = df_merged.merge(df_quality_out[quality_new_cols], on='ccn', how='left')

print(f'After merging staffing: {df_merged.shape}')

# Report coverage
print(f'\nNew feature coverage:')
new_features = ['rn_per_bed', 'physician_per_bed', 'clinical_staff_per_bed', 
                'rn_skill_mix', 'star_rating', 'quality_score',
                'mort_pct_worse', 'safety_pct_worse', 'readm_pct_worse']
for feat in new_features:
    if feat in df_merged.columns:
        n = df_merged[feat].notna().sum()
        print(f'  {feat}: {n:,} ({100*n/len(df_merged):.1f}%)')

# Save
output_path = OUTPUT_DIR / 'hospital_enriched_features.csv'
df_merged.to_csv(output_path, index=False)
print(f'\nSaved enriched dataset: {output_path}')
print(f'Shape: {df_merged.shape}')

Master dataset: 3,280 hospitals
After merging staffing: (3280, 87)

New feature coverage:
  rn_per_bed: 3,274 (99.8%)
  physician_per_bed: 3,274 (99.8%)
  clinical_staff_per_bed: 3,274 (99.8%)
  rn_skill_mix: 3,057 (93.2%)
  star_rating: 2,653 (80.9%)
  quality_score: 3,280 (100.0%)
  mort_pct_worse: 2,862 (87.3%)
  safety_pct_worse: 3,000 (91.5%)
  readm_pct_worse: 3,125 (95.3%)



Saved enriched dataset: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb10a_additional_features/hospital_enriched_features.csv
Shape: (3280, 87)


In [5]:
print('='*60)
print('ENRICHED FEATURE SUMMARY')
print('='*60)

print('\nStaffing Ratios (per bed):')
for col in ['rn_per_bed', 'lpn_per_bed', 'physician_per_bed', 'clinical_staff_per_bed', 'rn_skill_mix']:
    if col in df_merged.columns:
        s = df_merged[col].dropna()
        print(f'  {col}: mean={s.mean():.3f}, median={s.median():.3f}, std={s.std():.3f}, n={len(s):,}')

print('\nQuality Metrics:')
for col in ['star_rating', 'quality_score', 'mort_pct_worse', 'safety_pct_worse', 'readm_pct_worse']:
    if col in df_merged.columns:
        s = df_merged[col].dropna()
        print(f'  {col}: mean={s.mean():.2f}, median={s.median():.2f}, std={s.std():.2f}, n={len(s):,}')

print(f'\nTotal features: {len(df_merged.columns)}')
print(f'Total hospitals: {len(df_merged):,}')
print(f'\nNB10a COMPLETE')

ENRICHED FEATURE SUMMARY

Staffing Ratios (per bed):
  rn_per_bed: mean=1.277, median=1.087, std=3.149, n=3,274
  lpn_per_bed: mean=0.108, median=0.048, std=0.191, n=3,274
  physician_per_bed: mean=0.143, median=0.006, std=1.184, n=3,274
  clinical_staff_per_bed: mean=1.786, median=1.464, std=3.778, n=3,274
  rn_skill_mix: mean=0.901, median=0.952, std=0.144, n=3,057

Quality Metrics:


  star_rating: mean=3.08, median=3.00, std=1.10, n=2,653
  quality_score: mean=0.66, median=0.00, std=1.78, n=3,280
  mort_pct_worse: mean=2.75, median=0.00, std=8.52, n=2,862
  safety_pct_worse: mean=2.68, median=0.00, std=8.40, n=3,000
  readm_pct_worse: mean=7.41, median=0.00, std=11.33, n=3,125

Total features: 87
Total hospitals: 3,280

NB10a COMPLETE
